# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k24csaiml1e2411265-wq/ML_starter_FlyrankAI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Lane:** Content Refresh & Performance Prediction. **Decision moment:** 1 March 2026. Features use February 2026 observations plus historical query-mix information. No March outcome is used in the feature vector.

In [8]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn
import os, getpass, duckdb, pandas as pd, numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
HF_TOKEN=os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata; HF_TOKEN=userdata.get('HF_TOKEN')
    except Exception: pass
HF_TOKEN=HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
assert HF_TOKEN
con=duckdb.connect(); con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL='hf://datasets/FlyRank/internship-warehouse'
TABLES={'fact_daily':f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",'fact_query_90d':f"read_parquet('{REL}/fact_content_query_90d.parquet')"}
print('Connected to FlyRank warehouse.')

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.


In [9]:
features=con.sql(f"""
WITH base AS (
 SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) imp_prev30,SUM(gsc_clicks) clk_prev30,
 AVG(gsc_avg_position) pos_prev30,STDDEV_SAMP(gsc_avg_position) pos_volatility
 FROM {TABLES['fact_daily']}
 WHERE report_date>=DATE '2026-02-01' AND report_date<DATE '2026-03-01'
 GROUP BY 1,2 HAVING SUM(gsc_impressions)>=100),
queries AS (SELECT content_hash_id,SUM(impressions_90d) kept_impressions,MAX(impressions_90d) top_query_impressions FROM {TABLES['fact_query_90d']} GROUP BY content_hash_id)
SELECT b.*,CASE WHEN b.imp_prev30>0 THEN b.clk_prev30/b.imp_prev30 ELSE NULL END ctr_prev30,
CASE WHEN q.kept_impressions>0 THEN q.top_query_impressions/q.kept_impressions ELSE NULL END query_concentration
FROM base b LEFT JOIN queries q USING(content_hash_id)
""").df()
feature_cols=['imp_prev30','clk_prev30','ctr_prev30','pos_prev30','pos_volatility','query_concentration']
X_raw=features[feature_cols].copy(); imputer=SimpleImputer(strategy='median'); X_vector=pd.DataFrame(imputer.fit_transform(X_raw),columns=feature_cols,index=features.index)
scaler=StandardScaler(); X_scaled=pd.DataFrame(scaler.fit_transform(X_vector),columns=feature_cols,index=features.index)
print('Feature vector shape:',X_vector.shape); display(X_vector.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (80322, 6)


,imp_prev30,clk_prev30,ctr_prev30,pos_prev30,pos_volatility,query_concentration
0,2482.0,10.0,0.004029,7.596952,1.623466,0.097192
1,105.0,0.0,0.000000,24.212271,28.547146,0.448276
2,118.0,0.0,0.000000,59.069525,7.588668,0.128799
3,4125.0,4.0,0.000970,6.565319,1.125335,0.166898
4,138.0,0.0,0.000000,13.208852,11.762234,0.275362


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing | Available when? |
|---|---|---|---|
| `imp_prev30` | Previous-30-day impressions | Median | Before 1 Mar 2026 |
| `clk_prev30` | Previous-30-day clicks | Median | Before 1 Mar 2026 |
| `ctr_prev30` | Clicks / impressions in previous window | Median | Before 1 Mar 2026 |
| `pos_prev30` | Mean search position in previous window | Median | Before 1 Mar 2026 |
| `pos_volatility` | Variation in daily average position | Median | Before 1 Mar 2026 |
| `query_concentration` | Largest-query share of kept 90-day impressions | Median | Historical signal at decision time |

`client_hash_id` and `content_hash_id` are used for grouping/joining only, not as predictive inputs.

In [10]:
audit=pd.DataFrame({'feature':feature_cols,'missing_before_fill':[int(X_raw[c].isna().sum()) for c in feature_cols],'missing_after_fill':[int(X_vector[c].isna().sum()) for c in feature_cols],'future_window':[False]*len(feature_cols)})
display(audit)

,feature,missing_before_fill,missing_after_fill,future_window
0,imp_prev30,0,0,False
1,clk_prev30,0,0,False
2,ctr_prev30,0,0,False
3,pos_prev30,0,0,False
4,pos_volatility,5,0,False
5,query_concentration,14461,0,False


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The retained feature names are checked for leakage terms. Then one deliberately leaked feature is created from the March outcome and removed from the final vector.

In [11]:
leak_keywords=['future','label','target','outcome','decline','next30','march']
hits=[c for c in feature_cols if any(k in c.lower() for k in leak_keywords)]
print('Leakage keyword hits:',hits); assert not hits
print('Feature window: February 2026; decision moment: 1 March 2026.')

Leakage keyword hits: []
Feature window: February 2026; decision moment: 1 March 2026.


In [12]:
outcome=con.sql(f"""SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) imp_future30 FROM {TABLES['fact_daily']} WHERE report_date>=DATE '2026-03-01' AND report_date<DATE '2026-04-01' GROUP BY 1,2""").df()
attack=features.merge(outcome,on=['client_hash_id','content_hash_id'],how='inner')
attack['leak_future_decline']=(attack['imp_future30']<0.8*attack['imp_prev30']).astype(int)
print('Deliberate leaked column created:', 'leak_future_decline' in attack.columns)
print('It uses March outcome data and is unavailable at prediction time.')
X_final=pd.DataFrame(imputer.transform(features[feature_cols]),columns=feature_cols,index=features.index)
assert 'leak_future_decline' not in X_final.columns
print('Final vector columns:',list(X_final.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Deliberate leaked column created: True
It uses March outcome data and is unavailable at prediction time.
Final vector columns: ['imp_prev30', 'clk_prev30', 'ctr_prev30', 'pos_prev30', 'pos_volatility', 'query_concentration']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **Future/March impressions, clicks, CTR, position:** future-window leakage.
- **Label or target fields:** directly encode the prediction target.
- **`leak_future_decline`:** deliberately created for the leakage test only; removed afterward.
- **Client names, URLs, private queries:** privacy and unnecessary identifying detail.
- **Raw client/content IDs as model features:** identifier memorization risk; retained only for joins/grouping.
- **June 2026 `_sample` for feature development:** final-month outcome/test-window risk.

In [13]:
excluded=pd.DataFrame([
['future March performance','Future-window leakage'],['label/target fields','Label-derived leakage'],['leak_future_decline','Leakage test only'],['client names / URLs / private queries','Privacy / unnecessary detail'],['raw client/content IDs as model inputs','Identifier memorization risk'],['June 2026 _sample','Final-month outcome-window risk']],columns=['excluded_field_group','reason'])
display(excluded); print('Final feature count:',len(feature_cols))

,excluded_field_group,reason
0,future March performance,Future-window leakage
1,label/target fields,Label-derived leakage
2,leak_future_decline,Leakage test only
3,client names / URLs / private queries,Privacy / unnecessary detail
4,raw client/content IDs as model inputs,Identifier memorization risk
5,June 2026 _sample,Final-month outcome-window risk


Final feature count: 6


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.